In [1]:
!pip install transformers datasets torch accelerate gradio huggingface_hub -q
print("Setup complete!")


Setup complete!


In [2]:
from google.colab import files
uploaded = files.upload()


Saving dataset.txt.txt to dataset.txt.txt


In [6]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel, TextDataset, DataCollatorForLanguageModeling, Trainer, TrainingArguments

# 1) Tokenizer and model load
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained(model_name)

# 2) Making text dataset
train_dataset = TextDataset(
    tokenizer=tokenizer,
    file_path="dataset.txt",
    block_size=128,
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

# 3) Training settings
training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    overwrite_output_dir=True,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=1,
    logging_steps=50,
)

# 4) making trainer and doing trainning
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
)

trainer.train()

# 5) Fine‑tuned model saving
trainer.save_model("./gpt2-finetuned")
tokenizer.save_pretrained("./gpt2-finetuned")

print("Finetuning complete!")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


Finetuning complete!


In [11]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# Pad token set
def load_model(tokenizer_name, model_name):
    tokenizer = GPT2Tokenizer.from_pretrained(tokenizer_name)
    tokenizer.pad_token = tokenizer.eos_token
    model = GPT2LMHeadModel.from_pretrained(model_name)
    return model, tokenizer

# Models load
model_base, tokenizer_base = load_model("gpt2", "gpt2")
model_finetuned, tokenizer_finetuned = load_model("./gpt2-finetuned", "./gpt2-finetuned")

def generate_text(model, tokenizer, prompt, max_new_tokens=50, temperature=0.7):
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(
        inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True,
        top_k=40,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Tests
print("=== SUCCESS: Text Generation ===")
prompt1 = "Technology is changing"
print("1. Base GPT-2:", generate_text(model_base, tokenizer_base, prompt1))
print("\n2. Your Fine-tuned:", generate_text(model_finetuned, tokenizer_finetuned, prompt1))
print("\n3. Learning to code:", generate_text(model_finetuned, tokenizer_finetuned, "Learning to code"))


=== SUCCESS: Text Generation ===
1. Base GPT-2: Technology is changing the way people think about computing and is changing how it interacts with people."

At a news conference at the University of California, Berkeley, Calif., last month, former Google CEO Larry Page said that he's looking for a way to make the

2. Your Fine-tuned: Technology is changing the way we think. In this year's report, the World Economic Forum (WEF) said that "demand for consumer products has declined in the past three years in four key countries: Japan, China, India and the United States."



3. Learning to code: Learning to code in JavaScript is a relatively new experience. It's not a new experience for most developers, but it's definitely something you must learn before you can make a huge impact in any industry.

As I mentioned before, JavaScript is an open source language


In [12]:
import gradio as gr
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# Model load (already trained)
model = GPT2LMHeadModel.from_pretrained("./gpt2-finetuned")
tokenizer = GPT2Tokenizer.from_pretrained("./gpt2-finetuned")
tokenizer.pad_token = tokenizer.eos_token

def generate(prompt):
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(
        inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=60,
        temperature=0.7,
        do_sample=True,
        top_k=40,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Gradio interface
iface = gr.Interface(
    fn=generate,
    inputs=gr.Textbox(label="Enter prompt", placeholder="Technology is..."),
    outputs=gr.Textbox(label="Generated text"),
    title="🚀 Prodigy Task 1: GPT-2 Text Generator",
    description="Fine-tuned on custom tech dataset"
)

iface.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ecddbb64867b06a42a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://ecddbb64867b06a42a.gradio.live
